# `07_bicycle_route_infrastructure_per_side_metrics`: Per-side infrastructure metrics per spatial unit

## Introduction

### Purpose

This notebook computes per-side infrastructure metrics at three spatial levels (municipality, province, H3 cell) on the designated bicycle route network: facility kilometres broken down by `protection_level_per_side`, plus share metrics under two denominators ("of total" and "of classifiable"). Per the pipeline diagram and thesis Results, this is the stage-07 infrastructure-metric notebook for the designated branch, complementing `07_bicycle_route_extent_metrics` (network extent) and `07_non_bicycle_route_extent_metrics` (non-route scoping layer).

### What is being measured

The extent metrics in `07_bicycle_route_extent_metrics` answer how much designated route network exists in a spatial unit. The metrics here answer a different question: *of the designated route network that exists, what fraction is physically protected, unprotected, or unclassifiable?* The per-side classification produced in `05_bicycle_route_infrastructure_per_side` and clipped to spatial units in `06_bicycle_route_infrastructure_per_side_per_spatial_unit` is the input; this notebook aggregates the side-row table into one row per spatial unit with facility km by protection level and the corresponding shares (thesis §3.4.3, §3.4.4).

### Two denominators

Two denominator choices arise when computing shares:

- **`total_facility_km`** includes every side-row in the spatial unit (`protected`, `unprotected`, `no_infrastructure`, `separate_infrastructure`, `uncertain`, `mixed_traffic_uncertain`, `ferry`, `tagging_conflict`). Shares against this denominator answer: *of everything in the designated route network, how much is confirmed protected?* This is the conservative estimate of protection coverage.
- **`classifiable_facility_km`** = `protected + unprotected + no_infrastructure`: sides where infrastructure type is known with certainty. Shares against this denominator describe the quality of the described network but are blind to the share that cannot be classified from OSM data alone.

Both are computed and reported. Together they characterise OSM tagging completeness: a large gap between `pct_protected_of_total` and `pct_protected_of_classifiable` is the signature of incomplete tagging.

### Inputs

- `bicycle_route_infrastructure_per_side_per_municipality`, `_per_province`, `_per_h3_cell` from `06_bicycle_route_infrastructure_per_side_per_spatial_unit`, loaded from cache.
- `municipalities`, `provinces`, `h3_cells` from `03_boundaries_population`.
- `visualization_minmax_norm_interactive` from `functions.ipynb`.

### Outputs

| Spatial unit | Metrics table | Descriptive stats |
|---|---|---|
| Municipality | `bicycle_route_infrastructure_metrics_by_municipality` | `bicycle_route_infrastructure_stats_by_municipality` |
| Province | `bicycle_route_infrastructure_metrics_by_province` | `bicycle_route_infrastructure_stats_by_province` |
| H3 cell | `bicycle_route_infrastructure_metrics_by_h3_cell` | `bicycle_route_infrastructure_stats_by_h3_cell` |

### Dependencies on prior notebooks

- `06_bicycle_route_infrastructure_per_side_per_spatial_unit` (output read from cache rather than re-run).
- `03_boundaries_population.ipynb` (via `%run`): provides `municipalities`, `provinces`, `h3_cells`, plus `MaplibreBasemap` and `CartoStyle`.
- `functions.ipynb` (via `%run`): provides `visualization_minmax_norm_interactive`.

### Downstream consumers

- Stage 08 MAUP robustness check (province and H3 levels).
- Stage 09 *Synthesis*: combines these infrastructure metrics with extent metrics for the UNECE matrix application; the thesis-headline classifiability and protection results draw from the municipality-level metrics here.

### Table of contents

1. [Environment setup](#1-environment-setup)
2. [Core function](#2-core-function)
3. [Municipalities](#3-municipalities)
   - 3.1 [Compute metrics](#31-compute-metrics)
   - 3.2 [Descriptive statistics](#32-descriptive-statistics)
   - 3.3 [Visualisation](#33-visualisation)
4. [Provinces](#4-provinces)
   - 4.1 [Compute metrics](#41-compute-metrics)
   - 4.2 [Descriptive statistics](#42-descriptive-statistics)
   - 4.3 [Visualisation](#43-visualisation)
5. [H3 grid cells](#5-h3-grid-cells)
   - 5.1 [Compute metrics](#51-compute-metrics)
   - 5.2 [Descriptive statistics](#52-descriptive-statistics)
   - 5.3 [Visualisation](#53-visualisation)

---

## 1. Environment setup

### Libraries and extensions

In [1]:
import duckdb
import geopandas as gpd
import pandas as pd
import matplotlib.pyplot as plt
from IPython.utils import io

### Loading shared variables

The three spatial-unit tables (`municipalities`, `provinces`, `h3_cells`), plus `MaplibreBasemap` and `CartoStyle`, are brought in via `%run` of `03_boundaries_population`. The `visualization_minmax_norm_interactive` helper comes from `functions.ipynb`. The three per-side per-spatial-unit tables are loaded directly from the cached parquet outputs of `06_bicycle_route_infrastructure_per_side_per_spatial_unit`.

In [9]:
with io.capture_output() as captured:
    %run /home/vbo226/03_boundaries_population.ipynb
    %run /home/vbo226/functions.ipynb

bicycle_route_infrastructure_per_side_per_municipality = duckdb.read_parquet(
    "/local/data/vbo226/cache/bicycle_route_infrastructure_per_side_per_municipality.parquet"
)

bicycle_route_infrastructure_per_side_per_province = duckdb.read_parquet(
    "/local/data/vbo226/cache/bicycle_route_infrastructure_per_side_per_province.parquet"
)

bicycle_route_infrastructure_per_side_per_h3_cell = duckdb.read_parquet(
    "/local/data/vbo226/cache/bicycle_route_infrastructure_per_side_per_h3_cell.parquet"
)

In [10]:
# For each spatial unit (municipality, province and h3 cell), convert GeoDataFrame into Arrow for easy ingestion into DuckDB 
municipalities_arrow = municipalities.to_arrow()
provinces_arrow = provinces.to_arrow()
h3_cells_arrow = h3_cells.to_arrow()

---

## 2. Core function

`calculate_infrastructure_metrics_per_spatial_unit` aggregates a per-side per-spatial-unit table into one row per spatial unit. It pivots `protection_level_per_side` into columns, sums `clipped_length_meters` within each bucket, derives the two share denominators, and left-joins to the complete spatial-unit table so that units with no designated-route coverage appear in the output as zero-valued rows rather than being silently dropped.

`NULLIF` guards against division by zero for spatial units with zero `total_facility_km` or zero `classifiable_facility_km`; the corresponding share returns `NULL` rather than raising an error. The function is parameterised on the input table names and column names, so the same body handles municipalities, provinces, and H3 cells.

In [11]:
def calculate_infrastructure_metrics_per_spatial_unit(
    per_side_per_spatial_unit_table,
    spatial_unit_table,
    spatial_unit_code,
    spatial_unit_geometry,
    spatial_unit_areakm2,
    spatial_unit_population,
    spatial_unit_name=None,
):
    name_select = f"s.{spatial_unit_name}," if spatial_unit_name is not None else ""

    return duckdb.sql(f"""
    WITH aggregation AS (
        SELECT
            {spatial_unit_code},
            ROUND(SUM(CASE WHEN protection_level_per_side = 'protected'
                           THEN clipped_length_meters ELSE 0 END) / 1000, 3) AS protected_facility_km,
            ROUND(SUM(CASE WHEN protection_level_per_side = 'unprotected'
                           THEN clipped_length_meters ELSE 0 END) / 1000, 3) AS unprotected_facility_km,
            ROUND(SUM(CASE WHEN protection_level_per_side = 'no_infrastructure'
                           THEN clipped_length_meters ELSE 0 END) / 1000, 3) AS no_infrastructure_facility_km,
            ROUND(SUM(CASE WHEN protection_level_per_side = 'separate_infrastructure'
                           THEN clipped_length_meters ELSE 0 END) / 1000, 3) AS separate_infrastructure_facility_km,
            ROUND(SUM(CASE WHEN protection_level_per_side = 'uncertain'
                           THEN clipped_length_meters ELSE 0 END) / 1000, 3) AS uncertain_facility_km,
            ROUND(SUM(CASE WHEN protection_level_per_side = 'mixed_traffic_uncertain'
                           THEN clipped_length_meters ELSE 0 END) / 1000, 3) AS mixed_traffic_uncertain_facility_km,
            ROUND(SUM(clipped_length_meters) / 1000, 3) AS total_facility_km
        FROM {per_side_per_spatial_unit_table}
        GROUP BY {spatial_unit_code}
    ),
    with_shares AS (
        SELECT *,
            ROUND(protected_facility_km + unprotected_facility_km + no_infrastructure_facility_km, 3)
                AS classifiable_facility_km,
            ROUND(protected_facility_km * 100.0 / NULLIF(total_facility_km, 0), 2)
                AS pct_protected_of_total,
            ROUND(unprotected_facility_km * 100.0 / NULLIF(total_facility_km, 0), 2)
                AS pct_unprotected_of_total,
            ROUND(uncertain_facility_km * 100.0 / NULLIF(total_facility_km, 0), 2)
                AS pct_uncertain_of_total,
            ROUND(mixed_traffic_uncertain_facility_km * 100.0 / NULLIF(total_facility_km, 0), 2)
                AS pct_mixed_traffic_uncertain_of_total,
            ROUND(protected_facility_km * 100.0
                  / NULLIF(protected_facility_km + unprotected_facility_km + no_infrastructure_facility_km, 0), 2)
                AS pct_protected_of_classifiable,
            ROUND(unprotected_facility_km * 100.0
                  / NULLIF(protected_facility_km + unprotected_facility_km + no_infrastructure_facility_km, 0), 2)
                AS pct_unprotected_of_classifiable
        FROM aggregation
    )
    SELECT
        s.{spatial_unit_code},
        {name_select}
        s.{spatial_unit_geometry},
        s.{spatial_unit_areakm2},
        s.{spatial_unit_population},
        COALESCE(a.protected_facility_km,               0) AS protected_facility_km,
        COALESCE(a.unprotected_facility_km,             0) AS unprotected_facility_km,
        COALESCE(a.no_infrastructure_facility_km,       0) AS no_infrastructure_facility_km,
        COALESCE(a.separate_infrastructure_facility_km, 0) AS separate_infrastructure_facility_km,
        COALESCE(a.uncertain_facility_km,               0) AS uncertain_facility_km,
        COALESCE(a.mixed_traffic_uncertain_facility_km, 0) AS mixed_traffic_uncertain_facility_km,
        COALESCE(a.total_facility_km,                   0) AS total_facility_km,
        COALESCE(a.classifiable_facility_km,            0) AS classifiable_facility_km,
        a.pct_protected_of_total,
        a.pct_unprotected_of_total,
        a.pct_uncertain_of_total,
        a.pct_mixed_traffic_uncertain_of_total,
        a.pct_protected_of_classifiable,
        a.pct_unprotected_of_classifiable
    FROM {spatial_unit_table} s
    LEFT JOIN with_shares a
        ON s.{spatial_unit_code} = a.{spatial_unit_code}
    """)

---

## 3. Municipalities

### 3.1 Compute metrics

In [12]:
bicycle_route_infrastructure_metrics_by_municipality = calculate_infrastructure_metrics_per_spatial_unit(
    per_side_per_spatial_unit_table='bicycle_route_infrastructure_per_side_per_municipality',
    spatial_unit_table='municipalities_arrow',
    spatial_unit_code='municipality_code',
    spatial_unit_name='municipality_name',
    spatial_unit_geometry='geometry',
    spatial_unit_areakm2='area_km2',
    spatial_unit_population='population',
)

bicycle_route_infrastructure_metrics_by_municipality_gdf = gpd.GeoDataFrame.from_arrow(
    bicycle_route_infrastructure_metrics_by_municipality.arrow()
)

In [13]:
print(f"Municipalities: {len(bicycle_route_infrastructure_metrics_by_municipality_gdf):,}")

duckdb.sql("""
SELECT
    ROUND(SUM(protected_facility_km),               1) AS protected_facility_km,
    ROUND(SUM(unprotected_facility_km),             1) AS unprotected_facility_km,
    ROUND(SUM(no_infrastructure_facility_km),       1) AS no_infrastructure_facility_km,
    ROUND(SUM(uncertain_facility_km),               1) AS uncertain_facility_km,
    ROUND(SUM(mixed_traffic_uncertain_facility_km), 1) AS mixed_traffic_uncertain_facility_km,
    ROUND(SUM(total_facility_km),                   1) AS total_facility_km,
    ROUND(SUM(classifiable_facility_km),            1) AS classifiable_facility_km,
    ROUND(SUM(protected_facility_km) * 100.0
          / NULLIF(SUM(total_facility_km), 0), 2)       AS pct_protected_of_total,
    ROUND(SUM(protected_facility_km) * 100.0
          / NULLIF(SUM(classifiable_facility_km), 0), 2) AS pct_protected_of_classifiable
FROM bicycle_route_infrastructure_metrics_by_municipality
""")

Municipalities: 342


┌───────────────────────┬─────────────────────────┬───────────────────────────────┬───────────────────────┬─────────────────────────────────────┬───────────────────┬──────────────────────────┬────────────────────────┬───────────────────────────────┐
│ protected_facility_km │ unprotected_facility_km │ no_infrastructure_facility_km │ uncertain_facility_km │ mixed_traffic_uncertain_facility_km │ total_facility_km │ classifiable_facility_km │ pct_protected_of_total │ pct_protected_of_classifiable │
│        double         │         double          │            double             │        double         │               double                │      double       │          double          │         double         │            double             │
├───────────────────────┼─────────────────────────┼───────────────────────────────┼───────────────────────┼─────────────────────────────────────┼───────────────────┼──────────────────────────┼────────────────────────┼───────────────────────────────┤


### 3.2 Descriptive statistics

In [14]:
infra_metrics_municipality = [
    'pct_protected_of_total',
    'pct_protected_of_classifiable',
    'pct_uncertain_of_total',
    'pct_mixed_traffic_uncertain_of_total',
]

bicycle_route_infrastructure_stats_by_municipality = (
    bicycle_route_infrastructure_metrics_by_municipality_gdf[infra_metrics_municipality]
    .describe()
    .T
    .round(2)
)
bicycle_route_infrastructure_stats_by_municipality

,count,mean,std,min,25%,50%,75%,max
pct_protected_of_total,338.0,38.93,15.10,8.15,28.71,38.10,47.78,83.74
pct_protected_of_classifiable,338.0,72.42,17.71,12.18,61.52,75.10,85.80,100.00
pct_uncertain_of_total,338.0,0.92,1.75,0.00,0.00,0.22,0.78,12.30
pct_mixed_traffic_uncertain_of_total,338.0,44.48,16.17,10.65,31.96,45.13,56.85,85.16


`pct_protected_of_total` and `pct_protected_of_classifiable` will differ substantially for municipalities where `mixed_traffic_uncertain` accounts for a large share of the network. A municipality with a high `pct_protected_of_classifiable` but a low `pct_protected_of_total` is one where much of the designated route network lacks cycleway tags: the described portion is predominantly protected, but tagging coverage is incomplete. A municipality with similar values on both metrics has high tagging completeness.

`pct_uncertain_of_total` and `pct_mixed_traffic_uncertain_of_total` together characterise OSM tagging completeness for the municipality's designated route network. High values on either indicate that infrastructure type cannot be determined for a large share of the network from OSM data alone (thesis §3.4.4 inferred-absence categories).

### 3.3 Visualisation

#### Interactive

In [15]:
map_municipality_pct_protected_of_total = visualization_minmax_norm_interactive(
    bicycle_route_infrastructure_metrics_by_municipality_gdf, 'pct_protected_of_total'
)
map_municipality_pct_protected_of_classifiable = visualization_minmax_norm_interactive(
    bicycle_route_infrastructure_metrics_by_municipality_gdf, 'pct_protected_of_classifiable'
)
map_municipality_pct_uncertain_of_total = visualization_minmax_norm_interactive(
    bicycle_route_infrastructure_metrics_by_municipality_gdf, 'pct_uncertain_of_total'
)
map_municipality_pct_mixed_traffic_uncertain_of_total = visualization_minmax_norm_interactive(
    bicycle_route_infrastructure_metrics_by_municipality_gdf, 'pct_mixed_traffic_uncertain_of_total'
)

map_ = widgets.VBox([
    widgets.HBox([
        widgets.VBox([
            widgets.HTML("<h3 style='text-align:center;margin:0'>Protected share of total (%)</h3>"),
            map_municipality_pct_protected_of_total
        ], layout=widgets.Layout(flex="1")),
        widgets.VBox([
            widgets.HTML("<h3 style='text-align:center;margin:0'>Protected share of classifiable (%)</h3>"),
            map_municipality_pct_protected_of_classifiable
        ], layout=widgets.Layout(flex="1")),
    ]),
    widgets.HBox([
        widgets.VBox([
            widgets.HTML("<h3 style='text-align:center;margin:0'>Uncertain share of total (%)</h3>"),
            map_municipality_pct_uncertain_of_total
        ], layout=widgets.Layout(flex="1")),
        widgets.VBox([
            widgets.HTML("<h3 style='text-align:center;margin:0'>Mixed-traffic-uncertain share of total (%)</h3>"),
            map_municipality_pct_mixed_traffic_uncertain_of_total
        ], layout=widgets.Layout(flex="1")),
    ]),
])

display(map_)

Comparing `pct_protected_of_total` against `pct_mixed_traffic_uncertain_of_total` reveals the tagging-completeness gradient. Municipalities where `pct_protected_of_total` is high and `pct_mixed_traffic_uncertain_of_total` is low have a well-tagged, well-protected designated route network. Municipalities where `pct_protected_of_total` is low and `pct_mixed_traffic_uncertain_of_total` is high may have either genuinely unprotected infrastructure or simply incomplete tagging: `pct_protected_of_classifiable` is the disambiguator. A high `pct_protected_of_classifiable` with a high `pct_mixed_traffic_uncertain_of_total` signals incomplete tagging on a network that is protected where it is described; a low `pct_protected_of_classifiable` signals genuinely mixed-traffic provision.

---

## 4. Provinces

### 4.1 Compute metrics

In [16]:
bicycle_route_infrastructure_metrics_by_province = calculate_infrastructure_metrics_per_spatial_unit(
    per_side_per_spatial_unit_table='bicycle_route_infrastructure_per_side_per_province',
    spatial_unit_table='provinces_arrow',
    spatial_unit_code='province_code',
    spatial_unit_name='province_name',
    spatial_unit_geometry='geometry',
    spatial_unit_areakm2='area_km2',
    spatial_unit_population='population',
)

bicycle_route_infrastructure_metrics_by_province_gdf = gpd.GeoDataFrame.from_arrow(
    bicycle_route_infrastructure_metrics_by_province.arrow()
)

### 4.2 Descriptive statistics

In [17]:
infra_metrics_province = [
    'pct_protected_of_total',
    'pct_protected_of_classifiable',
    'pct_uncertain_of_total',
    'pct_mixed_traffic_uncertain_of_total',
]

bicycle_route_infrastructure_stats_by_province = (
    bicycle_route_infrastructure_metrics_by_province_gdf[infra_metrics_province]
    .describe()
    .T
    .round(2)
)
bicycle_route_infrastructure_stats_by_province

,count,mean,std,min,25%,50%,75%,max
pct_protected_of_total,12.0,37.24,10.05,22.54,31.27,35.41,41.98,61.11
pct_protected_of_classifiable,12.0,71.84,9.64,58.31,64.78,69.04,77.79,87.56
pct_uncertain_of_total,12.0,0.62,0.54,0.07,0.16,0.42,1.03,1.65
pct_mixed_traffic_uncertain_of_total,12.0,47.21,10.64,27.99,40.54,47.31,54.34,63.91


### 4.3 Visualisation

#### Interactive

In [18]:
map_province_pct_protected_of_total = visualization_minmax_norm_interactive(
    bicycle_route_infrastructure_metrics_by_province_gdf, 'pct_protected_of_total'
)
map_province_pct_protected_of_classifiable = visualization_minmax_norm_interactive(
    bicycle_route_infrastructure_metrics_by_province_gdf, 'pct_protected_of_classifiable'
)
map_province_pct_uncertain_of_total = visualization_minmax_norm_interactive(
    bicycle_route_infrastructure_metrics_by_province_gdf, 'pct_uncertain_of_total'
)
map_province_pct_mixed_traffic_uncertain_of_total = visualization_minmax_norm_interactive(
    bicycle_route_infrastructure_metrics_by_province_gdf, 'pct_mixed_traffic_uncertain_of_total'
)

map_ = widgets.VBox([
    widgets.HBox([
        widgets.VBox([
            widgets.HTML("<h3 style='text-align:center;margin:0'>Protected share of total (%)</h3>"),
            map_province_pct_protected_of_total
        ], layout=widgets.Layout(flex="1")),
        widgets.VBox([
            widgets.HTML("<h3 style='text-align:center;margin:0'>Protected share of classifiable (%)</h3>"),
            map_province_pct_protected_of_classifiable
        ], layout=widgets.Layout(flex="1")),
    ]),
    widgets.HBox([
        widgets.VBox([
            widgets.HTML("<h3 style='text-align:center;margin:0'>Uncertain share of total (%)</h3>"),
            map_province_pct_uncertain_of_total
        ], layout=widgets.Layout(flex="1")),
        widgets.VBox([
            widgets.HTML("<h3 style='text-align:center;margin:0'>Mixed-traffic-uncertain share of total (%)</h3>"),
            map_province_pct_mixed_traffic_uncertain_of_total
        ], layout=widgets.Layout(flex="1")),
    ]),
])

display(map_)

---

## 5. H3 grid cells

### 5.1 Compute metrics

In [20]:
bicycle_route_infrastructure_metrics_by_h3_cell = calculate_infrastructure_metrics_per_spatial_unit(
    per_side_per_spatial_unit_table='bicycle_route_infrastructure_per_side_per_h3_cell',
    spatial_unit_table='h3_cells_arrow',
    spatial_unit_code='h3_index',
    spatial_unit_geometry='geometry',
    spatial_unit_areakm2='area_km2',
    spatial_unit_population='population',
)

bicycle_route_infrastructure_metrics_by_h3_cell_gdf = gpd.GeoDataFrame.from_arrow(
    bicycle_route_infrastructure_metrics_by_h3_cell.arrow()
)

### 5.2 Descriptive statistics

In [21]:
infra_metrics_h3 = [
    'pct_protected_of_total',
    'pct_protected_of_classifiable',
    'pct_uncertain_of_total',
    'pct_mixed_traffic_uncertain_of_total',
]

bicycle_route_infrastructure_stats_by_h3_cell = (
    bicycle_route_infrastructure_metrics_by_h3_cell_gdf[infra_metrics_h3]
    .describe()
    .T
    .round(2)
)
bicycle_route_infrastructure_stats_by_h3_cell

,count,mean,std,min,25%,50%,75%,max
pct_protected_of_total,38985.0,37.03,40.91,0.0,0.00,17.65,80.54,100.0
pct_protected_of_classifiable,25818.0,79.04,37.28,0.0,74.90,100.00,100.00,100.0
pct_uncertain_of_total,38985.0,0.35,3.54,0.0,0.00,0.00,0.00,100.0
pct_mixed_traffic_uncertain_of_total,38985.0,52.21,42.10,0.0,1.34,52.34,100.00,100.0


### 5.3 Visualisation

#### Interactive

In [22]:
map_h3_pct_protected_of_total = visualization_minmax_norm_interactive(
    bicycle_route_infrastructure_metrics_by_h3_cell_gdf, 'pct_protected_of_total'
)
map_h3_pct_protected_of_classifiable = visualization_minmax_norm_interactive(
    bicycle_route_infrastructure_metrics_by_h3_cell_gdf, 'pct_protected_of_classifiable'
)
map_h3_pct_uncertain_of_total = visualization_minmax_norm_interactive(
    bicycle_route_infrastructure_metrics_by_h3_cell_gdf, 'pct_uncertain_of_total'
)
map_h3_pct_mixed_traffic_uncertain_of_total = visualization_minmax_norm_interactive(
    bicycle_route_infrastructure_metrics_by_h3_cell_gdf, 'pct_mixed_traffic_uncertain_of_total'
)

map_ = widgets.VBox([
    widgets.HBox([
        widgets.VBox([
            widgets.HTML("<h3 style='text-align:center;margin:0'>Protected share of total (%)</h3>"),
            map_h3_pct_protected_of_total
        ], layout=widgets.Layout(flex="1")),
        widgets.VBox([
            widgets.HTML("<h3 style='text-align:center;margin:0'>Protected share of classifiable (%)</h3>"),
            map_h3_pct_protected_of_classifiable
        ], layout=widgets.Layout(flex="1")),
    ]),
    widgets.HBox([
        widgets.VBox([
            widgets.HTML("<h3 style='text-align:center;margin:0'>Uncertain share of total (%)</h3>"),
            map_h3_pct_uncertain_of_total
        ], layout=widgets.Layout(flex="1")),
        widgets.VBox([
            widgets.HTML("<h3 style='text-align:center;margin:0'>Mixed-traffic-uncertain share of total (%)</h3>"),
            map_h3_pct_mixed_traffic_uncertain_of_total
        ], layout=widgets.Layout(flex="1")),
    ]),
])

display(map_)

/home/vbo226/.local/lib/python3.11/site-packages/lonboard/_geoarrow/ops/reproject.py:40: UserWarning: No CRS exists on data. If no data is shown on the map, double check that your CRS is WGS84.
  warn(
/home/vbo226/.local/lib/python3.11/site-packages/lonboard/_geoarrow/ops/reproject.py:40: UserWarning: No CRS exists on data. If no data is shown on the map, double check that your CRS is WGS84.
  warn(
/home/vbo226/.local/lib/python3.11/site-packages/lonboard/_geoarrow/ops/reproject.py:40: UserWarning: No CRS exists on data. If no data is shown on the map, double check that your CRS is WGS84.
  warn(
/home/vbo226/.local/lib/python3.11/site-packages/lonboard/_geoarrow/ops/reproject.py:40: UserWarning: No CRS exists on data. If no data is shown on the map, double check that your CRS is WGS84.
  warn(


### Export

Persist the six output tables to disk so the stage-08 MAUP and stage-09 synthesis notebooks can read them. The DuckDB relations use `write_parquet`; the pandas descriptive-statistics frames use `to_parquet`.

In [23]:
from pathlib import Path

Path("/local/data/vbo226/cache").mkdir(parents=True, exist_ok=True)

bicycle_route_infrastructure_metrics_by_municipality.write_parquet(
    "/local/data/vbo226/cache/bicycle_route_infrastructure_metrics_by_municipality.parquet"
)

bicycle_route_infrastructure_stats_by_municipality.to_parquet(
    "/local/data/vbo226/cache/bicycle_route_infrastructure_stats_by_municipality.parquet"
)

bicycle_route_infrastructure_metrics_by_province.write_parquet(
    "/local/data/vbo226/cache/bicycle_route_infrastructure_metrics_by_province.parquet"
)

bicycle_route_infrastructure_stats_by_province.to_parquet(
    "/local/data/vbo226/cache/bicycle_route_infrastructure_stats_by_province.parquet"
)

bicycle_route_infrastructure_metrics_by_h3_cell.write_parquet(
    "/local/data/vbo226/cache/bicycle_route_infrastructure_metrics_by_h3_cell.parquet"
)

bicycle_route_infrastructure_stats_by_h3_cell.to_parquet(
    "/local/data/vbo226/cache/bicycle_route_infrastructure_stats_by_h3_cell.parquet"
)